# 🎲 generation_config

既定 sampling 値が API にどう効くか、temperature=0 の扱い。


In [ ]:
import subprocess
import sys

print("=== 素の環境 ===")
r = subprocess.run([sys.executable, "-c",
                    "import torch, transformers; print(torch.__version__, transformers.__version__)"],
                   capture_output=True, text=True)
print("  ", (r.stdout or r.stderr).strip()[:200])

print("=== pip install vllm==0.32.0 ===")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input",
                    "vllm==0.32.0"], capture_output=True, text=True, timeout=5400)
tail = ((r.stdout or "") + (r.stderr or "")).strip().splitlines()
print("   rc:", r.returncode, "|", " / ".join(tail[-3:])[:300])
r = subprocess.run([sys.executable, "-c",
                    "import vllm, torch, transformers; print(vllm.__version__, torch.__version__, transformers.__version__)"],
                   capture_output=True, text=True)
print("   versions:", (r.stdout or r.stderr).strip()[:300])

def find_root():
    """dataset のマウント位置は 2 通りあるので config.json を探す。"""
    import glob
    slug = "qwen38-27b-uncensored-hauhaucs-awq-mtp"
    ordered = []
    for pat in ("/kaggle/input/*/config.json", "/kaggle/input/*/*/config.json",
                "/kaggle/input/*/*/*/config.json"):
        ordered += glob.glob(pat)
    cands = ([c for c in ordered if slug in c]
             or [c for c in ordered if "qwen" in c.lower()]
             or ordered)
    print("config.json 候補:", cands[:3])
    if not cands:
        print("/kaggle/input 直下:", glob.glob("/kaggle/input/*")[:10])
        raise SystemExit("dataset が見つからない")
    return cands[0].rsplit("/", 1)[0]

ROOT = find_root()

CASE = r"""
import inspect
import json
import sys

ROOT = sys.argv[1]
from vllm.config import ModelConfig
from vllm.sampling_params import SamplingParams

mc = ModelConfig(model=ROOT, dtype="half", max_model_len=8192,
                 trust_remote_code=True)
print("generation_config 設定:", mc.generation_config)
diff = mc.get_diff_sampling_param()
print("diff_sampling_param:", diff)

print("\n=== API 既定値との関係（serving 側の使い方）===")
src = inspect.getsource(SamplingParams.update_from_generation_config)
print(src[:1500])

print("\n=== SamplingParams の既定値 ===")
sp = SamplingParams()
for k in ("temperature", "top_p", "top_k", "repetition_penalty",
          "presence_penalty", "frequency_penalty", "seed", "stop_token_ids"):
    print(f"  {k} = {getattr(sp, k, '<missing>')!r}")

print("\n=== temperature=0 の扱い ===")
for t in (0.0, 0.01, None, 1.0):
    try:
        s = SamplingParams(temperature=t)
        print(f"  temperature={t!r} -> {s.temperature!r} greedy={getattr(s,'_greedy',None)}")
    except Exception as e:
        print(f"  temperature={t!r} -> {type(e).__name__}: {str(e)[:200]}")

print("\n=== generation_config を反映した SamplingParams ===")
sp = SamplingParams()
sp.update_from_generation_config(diff, eos_token_id=None)
for k in ("temperature", "top_p", "top_k", "repetition_penalty", "stop_token_ids"):
    print(f"  {k} = {getattr(sp, k, '<missing>')!r}")

print("\n=== eos 2 個の扱い ===")
gc = json.load(open(ROOT + "/generation_config.json"))
print("  generation_config:", gc)
sp = SamplingParams()
sp.update_from_generation_config(gc, eos_token_id=248046)
print("  stop_token_ids:", sp.stop_token_ids)
print("  all_stop_token_ids:", getattr(sp, "_all_stop_token_ids", None))
"""

import os
import subprocess
import sys
import tempfile

case = os.path.join(tempfile.gettempdir(), "gencase.py")
open(case, "w").write(CASE)
r = subprocess.run([sys.executable, case, ROOT], capture_output=True, text=True,
                   timeout=1800)
print((r.stdout or "")[-12000:])
if r.returncode != 0:
    print("=== stderr ===")
    print((r.stderr or "")[-3000:])
